In [2]:
import pandas as pd
import numpy as np
import pickle
from lightgbm import LGBMRegressor
MODELS_CACHE = {}

In [3]:
df = pd.read_csv("full_data.csv")

In [4]:
pitchers = list(df["pitcherId"].unique())
pitches = list(df["pitchType"].unique())
pitches.remove("KN")
pitches.remove("ST")
lr = ["L", "R"]

features = ['vertApprAngle', 'inducedVertBreak', 'horzBreak', 'horzApprAngle', 'extension', 'relX', 'relZ', 'releaseVelocity', 'spinRate', 'spinDir']
targets = ["rv"]

test = df[df["year"] == 2025]

In [5]:
def pred_rv(test, features, target, pitch, side):
    if (pitch, side) not in MODELS_CACHE:
        with open(f'models/25_{pitch}_{side}.pkl', 'rb') as f:
            MODELS_CACHE[(pitch, side)] = pickle.load(f)

    model = MODELS_CACHE[(pitch, side)]
    X_test = test[features]
    predictions = model.predict(X_test) 
    
    return pd.Series(predictions, index=test.index)

In [6]:
dfs = []

for pitcher in pitchers: 
    for pitch in pitches: 
        for side in lr: 
            te = test[(test["pitcherId"] == pitcher) & (test["pitchType"] == pitch) & (test["pitcherHand"] == side)] 
            if len(te) >= 10: 
                dfx = {} 
                dfx["Pitcher"] = te["Pitcher"].iloc[0]
                dfx["pitcherId"] = pitcher
                dfx["Team"] = te["pitchingTeam"].iloc[0]
                dfx["PitchType"] = pitch 
                dfx["PitcherThrows"] = side 
                dfx["Count"] = len(te) 
                for feature in features: 
                    dfx[feature] = te[feature].mean() 
                for target in targets: 
                    dfz = pd.DataFrame([dfx], columns=features) 
                    dfx["xrv"] = float(pred_rv(dfz, features, target, pitch, side).iloc[0]) 
                dfs.append(dfx)

In [76]:
stuff = pd.DataFrame(dfs).reset_index(drop=True)

In [8]:
stuff[stuff["Team"] == "URI"].sort_values("xrv")

,Pitcher,pitcherId,Team,PitchType,PitcherThrows,Count,vertApprAngle,inducedVertBreak,horzBreak,horzApprAngle,extension,relX,relZ,releaseVelocity,spinRate,spinDir,xrv
3001,P. Aikens,1299756340,URI,CU,L,10,-7.364154,4.032466,13.889763,3.975938,4.501587,3.384290,4.513831,72.575203,2590.269408,262.686994,-0.033854
2992,D. Asencio,1241946368,URI,SL,R,113,-7.322495,8.758297,-9.723717,-3.572629,5.581573,-1.844785,5.737151,76.541147,2252.073474,130.289475,0.007156
12981,N. Fletcher,1067176853,URI,CU,L,13,-9.338816,-7.508356,10.670626,2.358651,5.592543,1.441015,4.936968,68.590335,1681.128934,313.890374,0.010195
3012,T. Levesque,1214276096,URI,FA,L,563,-5.332769,21.542431,-6.344312,0.232201,5.851984,0.617154,5.934798,87.273640,2032.010137,162.681280,0.019905
1427,C. Johnston,1241946624,URI,SL,R,103,-9.114918,-5.549883,-17.808652,-3.380795,5.296686,-1.561890,6.053778,75.657785,2566.035418,75.751630,0.028282
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1429,C. Johnston,1241946624,URI,SI,R,12,-6.516243,10.274821,17.636866,0.482325,6.255370,-1.584437,6.012699,85.914307,2265.066130,243.207556,0.130937
1426,C. Johnston,1241946624,URI,FA,R,151,-6.025083,11.654134,12.638104,-0.313798,6.061297,-1.448858,6.116953,84.955957,2209.747747,229.432777,0.137534
3016,Z. Morris,1202393856,URI,FA,R,95,-5.365312,16.630204,12.699680,-0.604363,5.577461,-1.828187,5.757472,87.274775,2304.512224,218.801567,0.139560
12989,B. Hsu,1126083840,URI,CU,R,37,-8.920510,-12.299829,-10.228622,-2.147205,5.260421,-0.999712,5.113542,75.153439,1776.222329,35.284569,0.143876


In [77]:
from scipy import stats

#df['zscore'] = stats.zscore(df['column'])
stuff["Stuff+"] = 100 + (-10 * stuff.groupby("PitchType")["xrv"].transform(stats.zscore))
stuff["Stuff+"] = stuff["Stuff+"].round().astype(int)

In [10]:
stuff.sort_values("Stuff+", ascending=False)

,Pitcher,pitcherId,Team,PitchType,PitcherThrows,Count,vertApprAngle,inducedVertBreak,horzBreak,horzApprAngle,extension,relX,relZ,releaseVelocity,spinRate,spinDir,xrv,Stuff+
9176,S. Brady,1321144576,LMU,CU,L,140,-10.489160,-18.425456,18.441137,3.252646,5.644665,1.630285,5.728155,72.587248,3122.007927,318.115945,-0.103376,140
14301,J. Mitchell,1262857984,LIP,SL,L,192,-6.808401,9.163681,8.866529,3.927187,5.340532,2.732314,5.283333,77.811683,2528.254230,229.648945,-0.047385,138
3513,B. Cleaver,1318951936,UK,FC,L,54,-7.885573,0.996064,9.471774,4.102667,6.385784,2.824545,5.841386,77.608051,2132.319217,270.531095,-0.185594,136
16808,M. Walsh,1310820352,VCU,SL,L,72,-6.893353,1.748738,16.200957,5.812855,5.698515,4.350179,4.704096,77.148633,2769.594219,269.794363,-0.042602,136
15611,C. Kriebel,1263557376,FGCU,SL,L,138,-7.785183,1.558290,12.393571,3.542551,4.134506,2.573051,5.017280,76.153182,2432.053768,266.758863,-0.040842,136
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17139,C. Danz,1436355266,NW,SL,L,18,-3.619750,1.038481,-3.552298,-1.927680,5.816122,-0.714915,2.252179,65.121281,1268.289588,119.145511,0.208826,35
14035,B. Swindle,1208235163,JVST,SL,R,79,-7.339039,-0.907751,1.844646,2.345784,5.307560,1.688762,4.820428,81.799794,2413.357176,275.114296,0.215791,33
11838,A. Anderson,1123493376,NWST,CU,R,11,-5.755975,1.847771,-2.745304,0.331626,5.340767,-0.439775,3.255092,65.928381,1703.250994,170.344554,0.299249,30
12012,J. Lourens,1146595328,JKST,SL,L,43,-8.667477,1.235612,-8.075944,2.241270,4.578433,2.831879,5.398443,71.538700,1728.192238,118.668241,0.231483,26


In [11]:
stuff.to_csv("stuff.csv")

In [12]:
stuff[stuff["Count"] > 200].sort_values("Stuff+", ascending=False).hea

,Pitcher,pitcherId,Team,PitchType,PitcherThrows,Count,vertApprAngle,inducedVertBreak,horzBreak,horzApprAngle,extension,relX,relZ,releaseVelocity,spinRate,spinDir,xrv,Stuff+
13244,J. Volini,1320920832,FSU,CU,L,286,-9.847579,-17.443100,11.279419,2.371668,5.446581,1.046487,6.051180,77.516908,2766.827039,329.563768,-0.074178,132
1350,A. Sentlinger,1281698816,VT,FA,L,264,-4.197561,19.787803,-7.529650,0.400823,6.467114,1.316262,5.678269,92.349873,2324.997951,158.536852,-0.039292,130
2846,E. Norby,1335271936,ECU,SL,L,340,-7.443537,-1.123865,15.864906,5.013406,5.533849,3.206451,4.864266,79.439471,3043.516103,278.709807,-0.022379,128
1145,T. Santana,1176403456,CLMB,SL,R,361,-7.085936,8.266907,-14.084440,-3.375774,5.442251,-1.668868,5.625787,78.688127,2248.462193,117.510738,-0.021148,128
755,J. Tallon,1100869376,DUKE,FA,L,377,-4.201008,17.775375,-14.806244,-0.034907,6.546771,1.552238,5.418277,92.070336,2457.105064,138.012929,-0.025045,127
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8499,C. Mohan,1821920924,TTU,FA,R,284,-6.058764,16.045741,7.269246,-0.402195,5.401170,-0.901844,6.403218,92.184142,2288.135413,205.235923,0.205108,73
16246,B. Jacobs,1246857984,ASU,FS,L,210,-7.598726,6.327714,-9.456598,-0.315925,5.580122,0.884409,5.339922,82.370789,1284.076454,119.328186,0.405298,72
7857,J. Harris,1216810752,BYU,FA,R,242,-6.243614,11.477442,13.140017,0.803098,5.800649,-0.354034,5.871681,89.478725,1872.030851,230.291135,0.225663,68
1313,C. Cosper,1092635904,MER,SL,L,275,-8.632276,-5.001565,7.106461,2.937681,5.329313,2.345942,5.083604,74.927533,2062.132991,309.696018,0.131754,66


In [57]:
stuff[(stuff["Team"] == "URI") & (stuff["Count"] >= 11)].sort_values("Stuff+", ascending=False).head(60)

,Pitcher,pitcherId,Team,PitchType,PitcherThrows,Count,vertApprAngle,inducedVertBreak,horzBreak,horzApprAngle,extension,relX,relZ,releaseVelocity,spinRate,spinDir,xrv,Stuff+,weight,weighted_xrv
3012,T. Levesque,1214276096,URI,FA,L,563,-5.332769,21.542431,-6.344312,0.232201,5.851984,0.617154,5.934798,87.273640,2032.010137,162.681280,0.019905,117,0.547133,0.010890
2992,D. Asencio,1241946368,URI,SL,R,113,-7.322495,8.758297,-9.723717,-3.572629,5.581573,-1.844785,5.737151,76.541147,2252.073474,130.289475,0.007156,116,0.289003,0.002068
12987,B. Hsu,1126083840,URI,FA,R,135,-4.351926,20.420219,9.149534,-0.629127,5.988422,-0.900472,5.100162,86.600312,2107.138747,204.903186,0.043232,111,0.553279,0.023919
1419,J. Kopetski,1269533184,URI,FA,L,237,-5.634936,19.619824,-9.754404,1.046684,6.236890,1.589678,5.723711,87.520037,1832.293085,151.950531,0.045096,111,0.628647,0.028349
1430,C. Johnston,1241946624,URI,CH,R,42,-7.651950,6.223617,17.145982,-0.003781,6.239100,-1.470591,5.986494,80.320216,1799.291101,249.361731,0.039314,111,0.117647,0.004625
1416,E. Maloney,1085415168,URI,SI,R,12,-6.797850,13.242778,17.570871,1.702626,5.330377,-0.237006,6.790269,86.536849,2185.940369,235.657018,0.034249,110,0.260870,0.008934
1417,C. Maloney,1336247552,URI,FA,R,38,-5.931722,19.480657,8.535183,-1.895613,6.473471,-1.657670,5.595128,89.023493,2068.572096,205.152050,0.053580,109,0.550725,0.029508
3025,J. Sabbath,1110853376,URI,FA,R,270,-5.060699,18.632913,4.441349,-0.853437,6.262684,-0.979987,5.717739,90.401321,2236.969890,193.766746,0.050333,109,0.542169,0.027289
12981,N. Fletcher,1067176853,URI,CU,L,13,-9.338816,-7.508356,10.670626,2.358651,5.592543,1.441015,4.936968,68.590335,1681.128934,313.890374,0.010195,109,0.050781,0.000518
1427,C. Johnston,1241946624,URI,SL,R,103,-9.114918,-5.549883,-17.808652,-3.380795,5.296686,-1.561890,6.053778,75.657785,2566.035418,75.751630,0.028282,108,0.288515,0.008160


In [79]:
stuff[(stuff["Count"] > 50) & (stuff["PitchType"] == "FA")].sort_values("Stuff+", ascending=False).head(10)

,Pitcher,pitcherId,Team,PitchType,PitcherThrows,Count,vertApprAngle,inducedVertBreak,horzBreak,horzApprAngle,extension,relX,relZ,releaseVelocity,spinRate,spinDir,xrv,Stuff+
11657,J. Mayers,1307169536,LSU,FA,R,149,-5.464745,22.633968,7.231550,-1.149329,6.200066,-1.655203,7.111829,96.951614,2553.557814,198.416404,-0.037991,130
1350,A. Sentlinger,1281698816,VT,FA,L,264,-4.197561,19.787803,-7.529650,0.400823,6.467114,1.316262,5.678269,92.349873,2324.997951,158.536852,-0.039292,130
755,J. Tallon,1100869376,DUKE,FA,L,377,-4.201008,17.775375,-14.806244,-0.034907,6.546771,1.552238,5.418277,92.070336,2457.105064,138.012929,-0.025045,127
467,K. Emmons,1299809537,TOWS,FA,R,153,-4.717626,22.787661,9.717267,-0.746498,5.894912,-1.288321,6.211975,91.917438,2306.112495,204.569674,-0.024219,127
538,M. Gaither,1138681856,ULL,FA,L,102,-4.302027,18.683955,-18.165671,1.244674,4.877600,2.619864,5.378726,90.606004,2535.843004,133.648338,-0.023919,127
10325,S. Garcia,1602557566,ORE,FA,L,291,-4.749893,22.562727,-7.634308,0.833956,6.075337,1.368665,6.039528,91.938543,2306.112760,160.382131,-0.025897,127
13041,W. Sanford,1518897568,ORE,FA,R,457,-4.313728,22.106422,8.496320,-1.000238,6.583986,-1.572086,5.609768,91.443800,2459.161361,201.948129,-0.022509,126
9277,R. Ojeda,1142126848,UCI,FA,L,507,-4.448310,19.587006,-7.030891,1.182023,7.079689,1.425053,5.524121,90.889894,2235.526087,159.345437,-0.020634,126
11059,S. Fitzpatrick,1082403072,ASU,FA,L,126,-4.146151,13.229334,-16.085928,2.183028,6.369231,3.253769,4.593835,89.274559,2157.777802,126.971067,-0.019390,126
16581,T. Shaw,1885286845,GRAM,FA,L,56,-5.333288,21.479325,-3.094230,1.127771,6.869783,1.406667,6.231262,90.415476,1988.409607,171.216467,-0.018610,126


In [55]:
stuff[stuff["pitcherId"] == 1110853376]

,Pitcher,pitcherId,Team,PitchType,PitcherThrows,Count,vertApprAngle,inducedVertBreak,horzBreak,horzApprAngle,extension,relX,relZ,releaseVelocity,spinRate,spinDir,xrv,Stuff+,weight,weighted_xrv
3025,J. Sabbath,1110853376,URI,FA,R,270,-5.060699,18.632913,4.441349,-0.853437,6.262684,-0.979987,5.717739,90.401321,2236.969890,193.766746,0.050333,109,0.542169,0.027289
3026,J. Sabbath,1110853376,URI,SL,R,178,-7.877917,1.410960,-10.850299,-2.629685,5.670990,-1.125648,5.597775,78.802531,2295.931538,91.280971,0.032620,106,0.357430,0.011659
3027,J. Sabbath,1110853376,URI,CU,R,14,-8.736125,-9.259682,-14.505213,-3.190037,5.524936,-0.831898,5.668408,73.931028,2297.156233,53.441936,0.044072,100,0.028112,0.001239
3028,J. Sabbath,1110853376,URI,CH,R,36,-6.917030,5.981536,12.462454,-0.294691,6.285391,-1.356746,5.445818,84.104857,1616.569786,246.230320,0.120622,88,0.072289,0.008720


In [ ]:
rows = []
ids = list(stuff["pitcherId"].unique())

for i in ids:
    x = stuff[stuff["pitcherId"] == i]
    dx = {}
    dx["Pitcher"] = x["Pitcher"].iloc[0]
    dx["pitcherId"] = i
    dx["Team"] = x["Team"].iloc[0]
    dx["PitcherThrows"] = x["PitcherThrows"].iloc[0]
    for pt in list(stuff["PitchType"].unique()):
        if pt in list(x["PitchType"].unique()):
            dx["Stf+ " + pt] = x[x["PitchType"] == pt]["Stuff+"].iloc[0].astype("int64")
        else:
            dx["Stf+ " + pt] = np.nan
    rows.append(dx)

In [64]:
table = pd.DataFrame(rows).reset_index(drop=True)
stf_cols = [col for col in table.columns if col.startswith("Stf+")]
table[stf_cols] = table[stf_cols].astype("Int64")

In [51]:
total = stuff.groupby("pitcherId")["Count"].transform("sum")
stuff["weight"] = stuff["Count"] / total
stuff["weighted_xrv"] = stuff["xrv"] * stuff["weight"]

overall = stuff.groupby("pitcherId")["weighted_xrv"].sum().reset_index()
overall.columns = ["pitcherId", "overall_xrv"]

overall["Stuff+"] = (100 + (-10 * stats.zscore(overall["overall_xrv"]))).round().astype(int)

table = table.merge(overall[["pitcherId", "Stuff+"]], on="pitcherId", how="left")

In [74]:
table[table["Team"] == "UMBC"]

,Pitcher,pitcherId,Team,PitcherThrows,Stf+ FA,Stf+ SL,Stf+ CU,Stf+ FC,Stf+ SI,Stf+ CH,Stf+ FS
340,M. Cunningham,1279329792,UMBC,R,90,94,91,<NA>,<NA>,<NA>,<NA>
341,N. Remy,1310848000,UMBC,R,112,94,<NA>,116,<NA>,<NA>,100
343,S. Murphy,1138622464,UMBC,R,93,112,<NA>,<NA>,<NA>,<NA>,<NA>
344,Z. Kelly,1299084032,UMBC,R,94,99,105,<NA>,<NA>,<NA>,<NA>
345,E. Sargent,1162067200,UMBC,L,96,78,<NA>,97,42,102,<NA>
346,S. Bell,1235469056,UMBC,R,94,97,<NA>,103,<NA>,<NA>,100
438,L. Wiley,1223210240,UMBC,L,102,97,94,<NA>,<NA>,100,<NA>
1076,B. Fox,1866778115,UMBC,L,101,74,106,101,<NA>,<NA>,<NA>
1077,C. Ho,1162473728,UMBC,R,101,100,105,93,<NA>,<NA>,<NA>
1078,A. Denham,1181602304,UMBC,R,101,94,<NA>,<NA>,<NA>,<NA>,109


In [84]:
table[table["pitcherId"] == 1307169536]

,Pitcher,pitcherId,Team,PitcherThrows,Stf+ FA,Stf+ SL,Stf+ CU,Stf+ FC,Stf+ SI,Stf+ CH,Stf+ FS
3235,J. Mayers,1307169536,LSU,R,130,109,<NA>,<NA>,<NA>,<NA>,<NA>


In [83]:
pd.set_option('display.max_columns', None)
df[df["pitcherId"] == 1307169536].sort_values("relZ", ascending=False).head(10)

,Unnamed: 0,year,pitchingTeamId,pitchingTeam,conference,hittingTeamId,hittingTeam,gameId,gameDate,inning,balls,strikes,outs,pitcherId,Pitcher,pitcherHand,pitchType,plateLocX,plateLocZ,vertApprAngle,inducedVertBreak,horzBreak,horzApprAngle,extension,pitchResult,atbatResult,relX,relZ,releaseVelocity,spinRate,spinDir,rv
4012772,4012772,2024,730284032,NICH,D1 Southland Conference,3374,LT,4539508,2024-04-24 13:00:00,5,0,0,1,1307169536,J. Mayers,R,FA,-0.72145,0.33376,-8.431952,26.25206,2.33335,1.084220,5.15745,B,HR,0.10509,7.55541,93.05228,2461.661087,185.307216,0.13
4012770,4012770,2024,730284032,NICH,D1 Southland Conference,3374,LT,4539508,2024-04-24 13:00:00,5,1,2,0,1307169536,J. Mayers,R,FA,-0.09942,5.18122,-3.255412,24.86328,1.50984,0.376756,5.28923,B,K,0.12863,7.53764,95.49019,2463.796783,183.622289,0.13
4012775,4012775,2024,730284032,NICH,D1 Southland Conference,3374,LT,4539508,2024-04-24 13:00:00,5,1,2,1,1307169536,J. Mayers,R,FA,1.00845,4.72130,-3.802185,24.94316,0.39890,-0.914734,5.38323,B,HR,0.11772,7.53049,93.90292,2412.455954,180.954297,0.13
4012767,4012767,2024,730284032,NICH,D1 Southland Conference,3374,LT,4539508,2024-04-24 13:00:00,5,0,0,0,1307169536,J. Mayers,R,FA,0.76164,4.36465,-4.197598,25.78860,-2.35744,-1.051372,5.45427,B,K,-0.02638,7.51500,92.68573,2396.989014,174.546833,0.13
762127,762127,2025,5323,LSU,D1 Southeastern Conference,5380,UNO,163441134,2025-03-18 18:33:00,6,1,1,0,1307169536,J. Mayers,R,FA,-0.56202,4.48085,-4.125358,24.60127,7.49237,0.275663,5.94998,B,IP_OUT,-0.93010,7.49928,94.15018,2384.831528,197.761689,0.13
4561384,4561384,2024,730284032,NICH,D1 Southland Conference,3031,UCI,5336410,2024-05-31 13:00:00,1,1,0,0,1307169536,J. Mayers,R,FA,-1.39446,3.15185,-5.627129,23.44915,-1.32768,1.125710,5.40048,B,BB,-0.22890,7.49547,93.33306,2479.111004,176.603206,0.13
4012778,4012778,2024,730284032,NICH,D1 Southland Conference,3374,LT,4539508,2024-04-24 13:00:00,5,3,2,1,1307169536,J. Mayers,R,FA,0.17250,2.04332,-6.713619,24.24143,-1.37480,-0.040971,5.37119,IP,HR,0.24855,7.48704,93.71944,2423.720894,176.596544,1.76
4012773,4012773,2024,730284032,NICH,D1 Southland Conference,3374,LT,4539508,2024-04-24 13:00:00,5,1,0,1,1307169536,J. Mayers,R,FA,-0.15373,2.87798,-5.851156,24.74671,0.25106,0.164786,5.40325,SL,HR,-0.02023,7.48050,92.51050,2392.275371,180.609061,-0.15
4561386,4561386,2024,730284032,NICH,D1 Southland Conference,3031,UCI,5336410,2024-05-31 13:00:00,1,3,0,0,1307169536,J. Mayers,R,FA,-0.62713,2.18385,-6.736835,22.37100,-2.56169,0.188869,5.53479,SL,BB,-0.23722,7.47939,93.39470,2382.495573,173.131011,-0.15
4012768,4012768,2024,730284032,NICH,D1 Southland Conference,3374,LT,4539508,2024-04-24 13:00:00,5,1,0,0,1307169536,J. Mayers,R,FA,0.99919,2.31010,-6.331536,25.93588,2.16559,-1.047344,5.15205,SL,K,-0.16712,7.47459,92.54644,2470.213977,184.988639,-0.15
